<a href="https://colab.research.google.com/github/canurdon/SnowLine/blob/main/sentinel2_snow_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
import xml.etree.ElementTree as ET

ee.Authenticate()
ee.Initialize(project='snowline-491111')

In [ ]:
!pip install earthengine-api geemap -q

In [ ]:
#create Tantalus bounding box
aoi = ee.Geometry.Rectangle([-123.38, 49.75, -123.10, 49.95])

#Verify area loaded correctly
print('area of interest defined')
print('Bounds:', aoi.bounds().getInfo())

In [ ]:
# Load sentinel-2 surface reflectance collection
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
.filterBounds(aoi) \
.filterDate('2024-10-01', '2025-03-01') \
.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
.sort('system:time_start', False)

# Get most recent image
image = s2.first()

# Print some metadata
info = image.getInfo()
print('Image ID:', info['id'])
print('Date:', ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print('Cloud cover:', image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(), '%')

In [ ]:
# Select the bands we need
# Band 3 = Green, Band 11 =SWIR
green = image.select('B3')
swir = image.select('B11')

# Compute NDSI
# Formula: (Green - SWIR) / (Green + SWIR)
ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')

#Threshhold at 0.4 - standard snow detection cutoff
snow_mask = ndsi.gt(0.4)

# Visualise on an interactive map
map1 = geemap.Map()
map1.centerObject(aoi, zoom=11)

# Add layers
map1.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map1.addLayer(
    ndsi,
    {'min': -1, 'max': 1, 'palette': ['black', 'white']},
    'NDSI'
)
map1.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask'
)

map1


In [ ]:
# SCL identifies the type of terrain represented by each pixel
# Numbers we want to mask out:
# 3 – cloud shadow; 8 – cloud medium probability; 9 – cloud high probability; 10 – thin cirrus

# create variable for SCL
scl = image.select('SCL')

# create cloud mask: keep = 1; mask out = 0
cloud_mask = scl.neq(3) \
.And(scl.neq(8)) \
.And(scl.neq(9)) \
.And(scl.neq(10))

# Apply cloud mask to NDSI
ndsi_masked = ndsi.updateMask(cloud_mask)
snow_mask_masked = ndsi_masked.gt(0.4)

# Add to map
map2 = geemap.Map()
map2.centerObject(aoi, zoom=11)

map2.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map2.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask unmasked'
)
map2.addLayer(
    snow_mask_masked,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask masked'
)
map2.addLayer(
    cloud_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ffffff']},
    'Cloud mask'
)

map2

In [ ]:
import datetime

# Create 60 day window over which to assess each pixel
end_date = ee.Date(datetime.datetime.now())
start_date = end_date.advance(-60, 'day')

# Gather collection
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
.filterBounds(aoi) \
.filterDate(start_date, end_date)

# May need to add fiter for compute efficiency see DECISIONS.md

# Create unction to mask clouds and compute NDSI for each image
def process_image(image):
    scl = image.select('SCL')

    # Cloud mask
    cloud_mask = scl.neq(3) \
        .And(scl.neq(8)) \
        .And(scl.neq(9)) \
        .And(scl.neq(10))

    # Water mask
    water_mask = scl.neq(6)

    # Combined mask
    valid_mask = cloud_mask \
    .And(water_mask)

    # NDSI
    ndsi = image.normalizedDifference(['B3', 'B11']) \
               .rename('NDSI')

    # Snow mask
    snow = ndsi.gt(0.4).rename('snow')

    # Apply combined mask
    snow_masked = snow.updateMask(valid_mask)

    # Timestamp
    timestamp = image.metadata('system:time_start') \
                    .rename('timestamp') \
                    .clip(aoi)

    return snow_masked \
        .addBands(timestamp) \
        .clip(aoi) \
        .copyProperties(image, ['system:time_start'])

# Apply function to collection
processed = collection.map(process_image)

# Composite: most recent valid pixel
composite = processed.qualityMosaic('timestamp').clip(aoi)


print('composite band created')
print(' names:', composite.bandNames().getInfo())

In [ ]:
# Visualise the composite
map3 = geemap.Map()
map3.centerObject(aoi, zoom=11)

# Extract the snow band
snow_composite = composite.select('snow')

# Extract timestamp and convert to days ago
now_ms = ee.Date(datetime.datetime.now()).millis()
days_ago = composite.select('timestamp') \
.subtract(now_ms) \
.abs() \
.divide(86400000) \
.rename('days_ago')

map3.addLayer(
    snow_composite,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow composite'
)

map3.addLayer(
    days_ago,
    {'min': 0, 'max': 60, 'palette': ['00ff00', 'ffff00', 'ff0000']},
    'Pixel age (days)'
)

map3

In [ ]:
print('Composite geometry:', composite.geometry().getInfo()['type'])
print('Snow band extent:', composite.select('snow').geometry().getInfo()['type'])

In [ ]:
print(composite.geometry().getInfo()['coordinates'])

In [ ]:
import xml.etree.ElementTree as ET

# Parse GPX file
def parse_gpx(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()

    # Handle GPX namespace
    ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}

    coords = []
    elevations = []

    for trkpt in root.findall('.//gpx:trkpt', ns):
        lat = float(trkpt.get('lat'))
        lon = float(trkpt.get('lon'))
        ele = trkpt.find('gpx:ele', ns)
        elev = float(ele.text) if ele is not None else 0

        coords.append([lon, lat])
        elevations.append(elev)

    return coords, elevations

# Load the route
coords, elevations = parse_gpx('/content/tantalusFKT_2024.gpx')

print(f'Track points: {len(coords)}')
print(f'Start: {coords[0]}')
print(f'End: {coords[-1]}')
print(f'Min elevation: {min(elevations)}m')
print(f'Max elevation: {max(elevations)}m')

In [ ]:
# Simplify route — take every 50th point
# 21699 / 50 = ~434 points, plenty for 10m resolution
simplified = coords[::50]
print(f'Simplified to {len(simplified)} points')

# Create Earth Engine line geometry
route = ee.Geometry.LineString(simplified)

# Display on map with new AOI
map4 = geemap.Map()
map4.centerObject(route, zoom=11)

# Add snow composite
map4.addLayer(
    composite.select('snow'),
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow composite'
)

# Add route
map4.addLayer(
    ee.Image().paint(route, 1, 2),
    {'palette': ['ff0000']},
    'Tantalus Traverse'
)

map4

In [ ]:
# Sample snow values along the route
# Every 50m along the route, what is the snow value?
route_snow = composite.select('snow').reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=route,
    scale=10,
    maxPixels=1e9
)

print('Mean snow coverage along route:', route_snow.getInfo())

# Sample individual points along route for detailed analysis
sampled = composite.select('snow').sample(
    region=route,
    scale=10,
    numPixels=500,
    geometries=True
)

# Count snow vs non-snow points
snow_points = sampled.filter(ee.Filter.eq('snow', 1))
clear_points = sampled.filter(ee.Filter.eq('snow', 0))

total = sampled.size().getInfo()
snow_count = snow_points.size().getInfo()
clear_count = clear_points.size().getInfo()

print(f'Total sampled points: {total}')
print(f'Snow points: {snow_count}')
print(f'Clear points: {clear_count}')
print(f'Snow coverage: {round(snow_count/total*100, 1)}%')

In [ ]:
# Get coordinates and snow values for all sampled points
features = sampled.getInfo()['features']

# Extract into a simple list
points = []
for f in features:
    coords = f['geometry']['coordinates']
    snow = f['properties']['snow']
    points.append({
        'lon': coords[0],
        'lat': coords[1],
        'snow': snow
    })

# Sort by longitude as rough proxy for position along route
# Sigurd is west (-123.32), Alpha is east (-123.20)
points.sort(key=lambda x: x['lon'])

# Identify snow/clear transitions
segments = []
current_state = points[0]['snow']
segment_start = points[0]
segment_points = [points[0]]

for p in points[1:]:
    if p['snow'] != current_state:
        segments.append({
            'type': 'snow' if current_state == 1 else 'clear',
            'start_lon': segment_start['lon'],
            'start_lat': segment_start['lat'],
            'end_lon': p['lon'],
            'end_lat': p['lat'],
            'point_count': len(segment_points)
        })
        current_state = p['snow']
        segment_start = p
        segment_points = [p]
    else:
        segment_points.append(p)

# Print snow segments
print('Snow crossing segments:')
snow_segs = [s for s in segments if s['type'] == 'snow']
for i, s in enumerate(snow_segs):
    print(f'  Segment {i+1}: {s["start_lon"]:.4f} to {s["end_lon"]:.4f} ({s["point_count"]} points)')

In [ ]:
import math

def haversine(coord1, coord2):
    """Distance in metres between two [lon, lat] points"""
    R = 6371000
    lat1, lat2 = math.radians(coord1[1]), math.radians(coord2[1])
    dlat = math.radians(coord2[1] - coord1[1])
    dlon = math.radians(coord2[0] - coord1[0])
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Build list of points with cumulative distance along route
# Use original simplified coords as the route backbone
route_with_dist = []
cumulative_dist = 0
for i, coord in enumerate(simplified):
    if i > 0:
        cumulative_dist += haversine(simplified[i-1], coord)
    route_with_dist.append({
        'lon': coord[0],
        'lat': coord[1],
        'dist_m': cumulative_dist
    })

total_route_km = cumulative_dist / 1000
print(f'Total route length: {total_route_km:.1f} km')

# For each sampled point, find nearest route point and get its distance
def nearest_route_dist(point, route):
    min_dist = float('inf')
    nearest = None
    for rp in route:
        d = haversine([point['lon'], point['lat']], [rp['lon'], rp['lat']])
        if d < min_dist:
            min_dist = d
            nearest = rp
    return nearest['dist_m']

# Tag each sample point with distance along route
print('Calculating distances along route...')
for p in points:
    p['route_dist_m'] = nearest_route_dist(p, route_with_dist)

# Sort by distance along route
points.sort(key=lambda x: x['route_dist_m'])
print('Done')


In [ ]:
# Rebuild segments using route distance
MIN_SEGMENT_M = 200  # ignore snow patches shorter than 200m

segments_by_dist = []
current_state = points[0]['snow']
segment_start = points[0]
segment_points = [points[0]]

for p in points[1:]:
    if p['snow'] != current_state:
        dist_m = segment_points[-1]['route_dist_m'] - segment_start['route_dist_m']
        segments_by_dist.append({
            'type': 'snow' if current_state == 1 else 'clear',
            'start_m': segment_start['route_dist_m'],
            'end_m': segment_points[-1]['route_dist_m'],
            'start_lon': segment_start['lon'],
            'start_lat': segment_start['lat'],
            'end_lon': segment_points[-1]['lon'],
            'end_lat': segment_points[-1]['lat'],
            'length_m': dist_m,
            'point_count': len(segment_points)
        })
        current_state = p['snow']
        segment_start = p
        segment_points = [p]
    else:
        segment_points.append(p)

# Merge short segments into neighbours
def merge_short(segments, min_m):
    merged = []
    for seg in segments:
        if seg['length_m'] < min_m and merged:
            prev = merged[-1]
            prev['end_m'] = seg['end_m']
            prev['end_lon'] = seg['end_lon']
            prev['end_lat'] = seg['end_lat']
            prev['length_m'] = prev['end_m'] - prev['start_m']
            prev['point_count'] += seg['point_count']
        else:
            merged.append(dict(seg))
    return merged

# Run merge twice to catch cascading short segments
merged = merge_short(segments_by_dist, MIN_SEGMENT_M)
merged = merge_short(merged, MIN_SEGMENT_M)

# Print meaningful snow crossings
snow_segs = [s for s in merged if s['type'] == 'snow']
print(f'Snow crossings: {len(snow_segs)}')
print()
for i, s in enumerate(snow_segs):
    print(f'Crossing {i+1}:')
    print(f'  Start: {s["start_m"]/1000:.1f}km')
    print(f'  End:   {s["end_m"]/1000:.1f}km')
    print(f'  Length: {s["length_m"]/1000:.1f}km')
    print()